In [8]:
from pathlib import Path
import sys
import inspect
import json

current_path = Path.cwd().resolve()

PROJECT_ROOT = None

for path in [current_path] + list(current_path.parents):
    if (path / "src").is_dir() and (path / "models").is_dir():
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Fraud-detection-ML-V2 project root was not found."
    )

SRC_DIR = PROJECT_ROOT / "src"
MODELS_DIR = PROJECT_ROOT / "models"

paths_to_add = [str(PROJECT_ROOT), str(SRC_DIR)]

for path in paths_to_add:
    if path not in sys.path:
        sys.path.insert(0, path)

required_files = [
    SRC_DIR / "feature_contract.py",
    SRC_DIR / "ml_inference.py",
    SRC_DIR / "ml_service.py",
    SRC_DIR / "rule_engine.py",
    SRC_DIR / "decision_engine.py",
    SRC_DIR / "shap_explainer.py",
    SRC_DIR / "detection_service.py",
    MODELS_DIR / "xgboost_fraud_detector.json",
    MODELS_DIR / "feature_columns.json",
    MODELS_DIR / "model_info.json",
    MODELS_DIR / "deployment_config.json",
    MODELS_DIR / "rule_engine_config.json",
    MODELS_DIR / "decision_engine_config.json",
]

missing_files = [str(path) for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "Missing required files:\n" + "\n".join(missing_files)
    )

import ml_inference
import ml_service
import rule_engine
import decision_engine
import shap_explainer
import detection_service

modules = {
    "ml_inference": ml_inference,
    "ml_service": ml_service,
    "rule_engine": rule_engine,
    "decision_engine": decision_engine,
    "shap_explainer": shap_explainer,
    "detection_service": detection_service,
}

print("PHASE 12 PROJECT CHECK")
print("=" * 60)
print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print("=" * 60)

for module_name, module in modules.items():
    print(f"\n{module_name}")

    functions = inspect.getmembers(module, inspect.isfunction)

    public_functions = [
        (name, obj)
        for name, obj in functions
        if not name.startswith("_")
    ]

    if not public_functions:
        print("  No public functions found")
    else:
        for name, obj in public_functions:
            try:
                signature = inspect.signature(obj)
            except Exception:
                signature = "signature unavailable"

            print(f"  {name}{signature}")

print("\nENVIRONMENT CHECK: READY")

PHASE 12 PROJECT CHECK
Project root: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2
Python: 3.11.9

ml_inference
  predict_transaction(transaction)

ml_service
  get_ml_score(transaction)
  predict_transaction(transaction)

rule_engine
  amount_threshold_rule(transaction, config)
  evaluate_rules(transaction)
  impossible_travel_rule(transaction, config)
  merchant_mismatch_rule(transaction, config)
  velocity_rule(transaction, config)

decision_engine
  calculate_risk_score(ml_fraud_score, rule_score)
  classify_risk(risk_score)
  generate_decision_reason(ml_fraud_score, rule_flags, risk_score, decision)
  make_decision(ml_fraud_score, rule_result)

shap_explainer
  explain_transaction(transaction, top_n=3)

detection_service
  evaluate_rules(transaction)
  explain_transaction(transaction, top_n=3)
  feature_reason_text(feature, value, shap_value)
  generate_reason(rule_flags, ml_score, shap_result, decision)
  get_ml_score(transaction)
  make_decision(ml_fraud_score, rule_result)
  sc

In [9]:
benchmark_input = {
    "transaction_id": "BENCHMARK_000001",
    "user_id": "USER_000001",
    "amount": 100.0,
    "amount_vs_avg_ratio": 1.0,
    "txn_count_last_5min": 0,
    "time_since_last_txn_sec": 600.0,
    "distance_from_last_location_km": 0.0,
    "merchant_category_is_new_for_user": 0
}

decision_input = {
    "ml_fraud_score": 0.20,
    "rule_score": 0.25,
    "rule_flags": []
}

print("Benchmark Feature Vector")
print("=" * 60)
print(json.dumps(benchmark_input, indent=2))

print("\nDecision Engine Input")
print("=" * 60)
print(json.dumps(decision_input, indent=2))

Benchmark Feature Vector
{
  "transaction_id": "BENCHMARK_000001",
  "user_id": "USER_000001",
  "amount": 100.0,
  "amount_vs_avg_ratio": 1.0,
  "txn_count_last_5min": 0,
  "time_since_last_txn_sec": 600.0,
  "distance_from_last_location_km": 0.0,
  "merchant_category_is_new_for_user": 0
}

Decision Engine Input
{
  "ml_fraud_score": 0.2,
  "rule_score": 0.25,
  "rule_flags": []
}


In [10]:
from pathlib import Path
import sys
import inspect
import time
import numpy as np
import json

current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [current_path] + list(current_path.parents)
        if (path / "src").is_dir() and (path / "models").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Fraud-detection-ML-V2 project root was not found."
    )

SRC_DIR = PROJECT_ROOT / "src"

for path in [str(PROJECT_ROOT), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

import ml_inference
import ml_service

benchmark_input = {
    "transaction_id": "BENCHMARK_000001",
    "user_id": "USER_000001",
    "amount": 100.0,
    "amount_vs_avg_ratio": 1.0,
    "txn_count_last_5min": 0,
    "time_since_last_txn_sec": 600.0,
    "distance_from_last_location_km": 0.0,
    "merchant_category_is_new_for_user": 0
}

def get_public_functions(module):
    return {
        name: obj
        for name, obj in inspect.getmembers(module, inspect.isfunction)
        if not name.startswith("_")
    }

def find_ml_function():
    preferred_names = [
        "predict_transaction",
        "predict",
        "infer",
        "run_inference",
        "get_ml_score",
        "score_transaction"
    ]

    for module in [ml_inference, ml_service]:
        functions = get_public_functions(module)

        for name in preferred_names:
            if name in functions:
                return functions[name]

    return None

def prepare_call(function_obj, values):
    signature = inspect.signature(function_obj)
    parameters = list(signature.parameters.values())

    if len(parameters) == 1:
        parameter = parameters[0]

        if parameter.kind in (
            inspect.Parameter.POSITIONAL_ONLY,
            inspect.Parameter.POSITIONAL_OR_KEYWORD
        ):
            parameter_name = parameter.name

            if parameter_name in values:
                return (values[parameter_name],), {}

            return (values,), {}

    kwargs = {}

    for parameter in parameters:
        if parameter.kind in (
            inspect.Parameter.VAR_POSITIONAL,
            inspect.Parameter.VAR_KEYWORD
        ):
            continue

        if parameter.name in values:
            kwargs[parameter.name] = values[parameter.name]
        elif parameter.default is inspect.Parameter.empty:
            raise ValueError(
                f"Required parameter '{parameter.name}' could not be supplied."
            )

    return (), kwargs

ml_function = find_ml_function()

if ml_function is None:
    raise RuntimeError(
        "No supported ML inference function was found in ml_inference.py or ml_service.py."
    )

args, kwargs = prepare_call(ml_function, benchmark_input)

warmup_runs = 20
benchmark_runs = 1000

for _ in range(warmup_runs):
    ml_function(*args, **kwargs)

latencies_ms = []

for _ in range(benchmark_runs):
    start = time.perf_counter()
    ml_function(*args, **kwargs)
    end = time.perf_counter()

    latencies_ms.append((end - start) * 1000)

latencies_ms = np.asarray(latencies_ms)

results = {
    "component": "ML inference",
    "function": ml_function.__name__,
    "warmup_runs": warmup_runs,
    "benchmark_runs": benchmark_runs,
    "mean_ms": float(np.mean(latencies_ms)),
    "median_ms": float(np.median(latencies_ms)),
    "p95_ms": float(np.percentile(latencies_ms, 95)),
    "p99_ms": float(np.percentile(latencies_ms, 99)),
    "min_ms": float(np.min(latencies_ms)),
    "max_ms": float(np.max(latencies_ms))
}

print(json.dumps(results, indent=2))

{
  "component": "ML inference",
  "function": "predict_transaction",
  "warmup_runs": 20,
  "benchmark_runs": 1000,
  "mean_ms": 4.043067800000699,
  "median_ms": 3.67515000016283,
  "p95_ms": 5.997874999911801,
  "p99_ms": 7.248611999807506,
  "min_ms": 2.980400000069494,
  "max_ms": 11.708600000019942
}


In [11]:
from pathlib import Path
import sys
import inspect
import time
import numpy as np
import json

current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [current_path] + list(current_path.parents)
        if (path / "src").is_dir() and (path / "models").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Fraud-detection-ML-V2 project root was not found."
    )

SRC_DIR = PROJECT_ROOT / "src"

for path in [str(PROJECT_ROOT), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

import rule_engine

benchmark_input = {
    "transaction_id": "BENCHMARK_000001",
    "user_id": "USER_000001",
    "amount": 100.0,
    "amount_vs_avg_ratio": 1.0,
    "txn_count_last_5min": 0,
    "time_since_last_txn_sec": 600.0,
    "distance_from_last_location_km": 0.0,
    "merchant_category_is_new_for_user": 0
}

def get_public_functions(module):
    return {
        name: obj
        for name, obj in inspect.getmembers(module, inspect.isfunction)
        if not name.startswith("_")
    }

def find_rule_function():
    preferred_names = [
        "evaluate_rules",
        "run_rules",
        "apply_rules",
        "check_rules",
        "detect_rules"
    ]

    functions = get_public_functions(rule_engine)

    for name in preferred_names:
        if name in functions:
            return functions[name]

    candidates = [
        (name, obj)
        for name, obj in functions.items()
        if "rule" in name.lower()
    ]

    if candidates:
        return candidates[0][1]

    return None

def prepare_call(function_obj, values):
    signature = inspect.signature(function_obj)
    parameters = list(signature.parameters.values())

    if len(parameters) == 1:
        parameter = parameters[0]

        if parameter.kind in (
            inspect.Parameter.POSITIONAL_ONLY,
            inspect.Parameter.POSITIONAL_OR_KEYWORD
        ):
            if parameter.name in values:
                return (values[parameter.name],), {}

            return (values,), {}

    kwargs = {}

    for parameter in parameters:
        if parameter.kind in (
            inspect.Parameter.VAR_POSITIONAL,
            inspect.Parameter.VAR_KEYWORD
        ):
            continue

        if parameter.name in values:
            kwargs[parameter.name] = values[parameter.name]
        elif parameter.default is inspect.Parameter.empty:
            raise ValueError(
                f"Required parameter '{parameter.name}' could not be supplied."
            )

    return (), kwargs

rule_function = find_rule_function()

if rule_function is None:
    raise RuntimeError(
        "No supported Rule Engine function was found in rule_engine.py."
    )

args, kwargs = prepare_call(rule_function, benchmark_input)

warmup_runs = 20
benchmark_runs = 1000

for _ in range(warmup_runs):
    rule_function(*args, **kwargs)

latencies_ms = []

for _ in range(benchmark_runs):
    start = time.perf_counter()
    rule_function(*args, **kwargs)
    end = time.perf_counter()

    latencies_ms.append((end - start) * 1000)

latencies_ms = np.asarray(latencies_ms)

results = {
    "component": "Rule Engine",
    "function": rule_function.__name__,
    "warmup_runs": warmup_runs,
    "benchmark_runs": benchmark_runs,
    "mean_ms": float(np.mean(latencies_ms)),
    "median_ms": float(np.median(latencies_ms)),
    "p95_ms": float(np.percentile(latencies_ms, 95)),
    "p99_ms": float(np.percentile(latencies_ms, 99)),
    "min_ms": float(np.min(latencies_ms)),
    "max_ms": float(np.max(latencies_ms))
}

print(json.dumps(results, indent=2))

{
  "component": "Rule Engine",
  "function": "evaluate_rules",
  "warmup_runs": 20,
  "benchmark_runs": 1000,
  "mean_ms": 0.002790900000036345,
  "median_ms": 0.002499999936844688,
  "p95_ms": 0.004105000289200684,
  "p99_ms": 0.005702000198652966,
  "min_ms": 0.002100000074278796,
  "max_ms": 0.016700000287528383
}


In [13]:
from pathlib import Path
import sys
import inspect
import time
import json
import numpy as np

current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [current_path] + list(current_path.parents)
        if (path / "src").is_dir() and (path / "models").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Fraud-detection-ML-V2 project root was not found."
    )

SRC_DIR = PROJECT_ROOT / "src"

for path in [str(PROJECT_ROOT), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

import rule_engine
import decision_engine

benchmark_input = {
    "transaction_id": "BENCHMARK_000001",
    "user_id": "USER_000001",
    "amount": 100.0,
    "amount_vs_avg_ratio": 1.0,
    "txn_count_last_5min": 0,
    "time_since_last_txn_sec": 600.0,
    "distance_from_last_location_km": 0.0,
    "merchant_category_is_new_for_user": 0
}

rule_functions = {
    name: obj
    for name, obj in inspect.getmembers(rule_engine, inspect.isfunction)
    if not name.startswith("_")
}

decision_functions = {
    name: obj
    for name, obj in inspect.getmembers(decision_engine, inspect.isfunction)
    if not name.startswith("_")
}

rule_function = None

for name in [
    "evaluate_rules",
    "run_rules",
    "apply_rules",
    "check_rules",
    "detect_rules"
]:
    if name in rule_functions:
        rule_function = rule_functions[name]
        break

if rule_function is None:
    candidates = [
        obj
        for name, obj in rule_functions.items()
        if "rule" in name.lower()
    ]

    if candidates:
        rule_function = candidates[0]

if rule_function is None:
    raise RuntimeError(
        "No Rule Engine function was found in rule_engine.py."
    )

decision_function = None

for name in [
    "calculate_decision",
    "make_decision",
    "evaluate_decision",
    "get_decision",
    "decide",
    "calculate_risk"
]:
    if name in decision_functions:
        decision_function = decision_functions[name]
        break

if decision_function is None:
    candidates = [
        obj
        for name, obj in decision_functions.items()
        if any(
            word in name.lower()
            for word in ["decision", "risk"]
        )
    ]

    if candidates:
        decision_function = candidates[0]

if decision_function is None:
    raise RuntimeError(
        "No Decision Engine function was found in decision_engine.py."
    )

def call_rule_function(function_obj, values):
    signature = inspect.signature(function_obj)
    parameters = list(signature.parameters.values())

    if len(parameters) == 1:
        parameter = parameters[0]

        if parameter.name in values:
            return function_obj(values[parameter.name])

        return function_obj(values)

    kwargs = {}

    for parameter in parameters:
        if parameter.kind in (
            inspect.Parameter.VAR_POSITIONAL,
            inspect.Parameter.VAR_KEYWORD
        ):
            continue

        if parameter.name in values:
            kwargs[parameter.name] = values[parameter.name]
        elif parameter.default is inspect.Parameter.empty:
            raise ValueError(
                f"Required Rule Engine parameter '{parameter.name}' could not be supplied."
            )

    return function_obj(**kwargs)

rule_result = call_rule_function(
    rule_function,
    benchmark_input
)

decision_values = {
    "rule_result": rule_result,
    "ml_fraud_score": 0.20,
    "rule_score": (
        rule_result.get("rule_score", 0.0)
        if isinstance(rule_result, dict)
        else 0.0
    ),
    "rule_flags": (
        rule_result.get("rule_flags", [])
        if isinstance(rule_result, dict)
        else []
    )
}

decision_signature = inspect.signature(decision_function)
decision_kwargs = {}

for parameter in decision_signature.parameters.values():
    if parameter.kind in (
        inspect.Parameter.VAR_POSITIONAL,
        inspect.Parameter.VAR_KEYWORD
    ):
        continue

    if parameter.name in decision_values:
        decision_kwargs[parameter.name] = decision_values[parameter.name]
    elif parameter.default is inspect.Parameter.empty:
        raise ValueError(
            f"Required Decision Engine parameter '{parameter.name}' "
            f"could not be supplied."
        )

decision_function(**decision_kwargs)

warmup_runs = 20
benchmark_runs = 1000

for _ in range(warmup_runs):
    decision_function(**decision_kwargs)

latencies_ms = []

for _ in range(benchmark_runs):
    start = time.perf_counter()

    decision_function(**decision_kwargs)

    end = time.perf_counter()

    latencies_ms.append((end - start) * 1000)

latencies_ms = np.asarray(latencies_ms)

results = {
    "component": "Decision Engine",
    "function": decision_function.__name__,
    "rule_function": rule_function.__name__,
    "warmup_runs": warmup_runs,
    "benchmark_runs": benchmark_runs,
    "mean_ms": float(np.mean(latencies_ms)),
    "median_ms": float(np.median(latencies_ms)),
    "p95_ms": float(np.percentile(latencies_ms, 95)),
    "p99_ms": float(np.percentile(latencies_ms, 99)),
    "min_ms": float(np.min(latencies_ms)),
    "max_ms": float(np.max(latencies_ms)),
    "rule_result": rule_result,
    "decision_input": decision_kwargs
}

print(json.dumps(results, indent=2, default=str))

{
  "component": "Decision Engine",
  "function": "make_decision",
  "rule_function": "evaluate_rules",
  "warmup_runs": 20,
  "benchmark_runs": 1000,
  "mean_ms": 0.014765599994007061,
  "median_ms": 0.010600000223348616,
  "p95_ms": 0.03320499995425052,
  "p99_ms": 0.07472600036180663,
  "min_ms": 0.006599999778700294,
  "max_ms": 0.20000000040454324,
  "rule_result": {
    "rule_flags": [],
    "rule_score": 0.0,
    "rules_triggered": 0
  },
  "decision_input": {
    "ml_fraud_score": 0.2,
    "rule_result": {
      "rule_flags": [],
      "rule_score": 0.0,
      "rules_triggered": 0
    }
  }
}


In [14]:
from pathlib import Path
import sys
import inspect
import time
import numpy as np
import json

current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [current_path] + list(current_path.parents)
        if (path / "src").is_dir() and (path / "models").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Fraud-detection-ML-V2 project root was not found."
    )

SRC_DIR = PROJECT_ROOT / "src"

for path in [str(PROJECT_ROOT), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

import detection_service

benchmark_input = {
    "transaction_id": "BENCHMARK_000001",
    "user_id": "USER_000001",
    "amount": 100.0,
    "amount_vs_avg_ratio": 1.0,
    "txn_count_last_5min": 0,
    "time_since_last_txn_sec": 600.0,
    "distance_from_last_location_km": 0.0,
    "merchant_category_is_new_for_user": 0
}

def get_public_functions(module):
    return {
        name: obj
        for name, obj in inspect.getmembers(module, inspect.isfunction)
        if not name.startswith("_")
    }

def find_detection_function():
    preferred_names = [
        "detect_transaction",
        "detect",
        "process_transaction",
        "run_detection",
        "process_detection"
    ]

    functions = get_public_functions(detection_service)

    for name in preferred_names:
        if name in functions:
            return functions[name]

    candidates = [
        (name, obj)
        for name, obj in functions.items()
        if any(
            word in name.lower()
            for word in ["detect", "process", "score"]
        )
    ]

    if candidates:
        return candidates[0][1]

    return None

def prepare_call(function_obj, values):
    signature = inspect.signature(function_obj)
    parameters = list(signature.parameters.values())

    if len(parameters) == 1:
        parameter = parameters[0]

        if parameter.kind in (
            inspect.Parameter.POSITIONAL_ONLY,
            inspect.Parameter.POSITIONAL_OR_KEYWORD
        ):
            if parameter.name in values:
                return (values[parameter.name],), {}

            return (values,), {}

    kwargs = {}

    for parameter in parameters:
        if parameter.kind in (
            inspect.Parameter.VAR_POSITIONAL,
            inspect.Parameter.VAR_KEYWORD
        ):
            continue

        if parameter.name in values:
            kwargs[parameter.name] = values[parameter.name]
        elif parameter.default is inspect.Parameter.empty:
            raise ValueError(
                f"Required parameter '{parameter.name}' could not be supplied."
            )

    return (), kwargs

detection_function = find_detection_function()

if detection_function is None:
    raise RuntimeError(
        "No supported detection function was found in detection_service.py."
    )

warmup_runs = 20
benchmark_runs = 1000

warmup_args, warmup_kwargs = prepare_call(
    detection_function,
    benchmark_input
)

for _ in range(warmup_runs):
    detection_function(*warmup_args, **warmup_kwargs)

latencies_ms = []
outputs = []

for i in range(benchmark_runs):
    transaction = benchmark_input.copy()
    transaction["transaction_id"] = f"BENCHMARK_{i:06d}"

    args, kwargs = prepare_call(
        detection_function,
        transaction
    )

    start = time.perf_counter()

    output = detection_function(*args, **kwargs)

    end = time.perf_counter()

    outputs.append(output)
    latencies_ms.append((end - start) * 1000)

latencies_ms = np.asarray(latencies_ms)

results = {
    "component": "Full detection pipeline",
    "function": detection_function.__name__,
    "warmup_runs": warmup_runs,
    "benchmark_runs": benchmark_runs,
    "mean_ms": float(np.mean(latencies_ms)),
    "median_ms": float(np.median(latencies_ms)),
    "p95_ms": float(np.percentile(latencies_ms, 95)),
    "p99_ms": float(np.percentile(latencies_ms, 99)),
    "min_ms": float(np.min(latencies_ms)),
    "max_ms": float(np.max(latencies_ms))
}

print(json.dumps(results, indent=2))

{
  "component": "Full detection pipeline",
  "function": "get_ml_score",
  "warmup_runs": 20,
  "benchmark_runs": 1000,
  "mean_ms": 4.0761532999886185,
  "median_ms": 3.7378499998794723,
  "p95_ms": 5.86966000009852,
  "p99_ms": 7.996756999832542,
  "min_ms": 2.9538999997384963,
  "max_ms": 55.26349999990998
}


In [16]:
from pathlib import Path
import sys
import inspect
import time
import json
import numpy as np

current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [current_path] + list(current_path.parents)
        if (path / "src").is_dir() and (path / "models").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Fraud-detection-ML-V2 project root was not found."
    )

SRC_DIR = PROJECT_ROOT / "src"

for path in [str(PROJECT_ROOT), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

import detection_service

benchmark_input = {
    "transaction_id": "BENCHMARK_000001",
    "user_id": "USER_000001",
    "amount": 100.0,
    "amount_vs_avg_ratio": 1.0,
    "txn_count_last_5min": 0,
    "time_since_last_txn_sec": 600.0,
    "distance_from_last_location_km": 0.0,
    "merchant_category_is_new_for_user": 0
}

public_functions = {
    name: obj
    for name, obj in inspect.getmembers(
        detection_service,
        inspect.isfunction
    )
    if not name.startswith("_")
}

preferred_names = [
    "detect_transaction",
    "detect",
    "process_transaction",
    "run_detection",
    "process_detection"
]

detection_function = None

for name in preferred_names:
    if name in public_functions:
        detection_function = public_functions[name]
        break

if detection_function is None:
    candidates = [
        obj
        for name, obj in public_functions.items()
        if any(
            word in name.lower()
            for word in ["detect", "process", "score"]
        )
    ]

    if candidates:
        detection_function = candidates[0]

if detection_function is None:
    raise RuntimeError(
        "Could not identify the production detection function "
        "inside src/detection_service.py."
    )

def call_detection(function_obj, values):
    signature = inspect.signature(function_obj)
    parameters = list(signature.parameters.values())

    if len(parameters) == 1:
        parameter = parameters[0]

        if parameter.name in values:
            return function_obj(values[parameter.name])

        return function_obj(values)

    kwargs = {}

    for parameter in parameters:
        if parameter.kind in (
            inspect.Parameter.VAR_POSITIONAL,
            inspect.Parameter.VAR_KEYWORD
        ):
            continue

        if parameter.name in values:
            kwargs[parameter.name] = values[parameter.name]
        elif parameter.default is inspect.Parameter.empty:
            raise ValueError(
                f"Required parameter '{parameter.name}' "
                f"could not be supplied."
            )

    return function_obj(**kwargs)

warmup_runs = 20
benchmark_runs = 1000

for _ in range(warmup_runs):
    call_detection(
        detection_function,
        benchmark_input
    )

latencies_ms = []
sample_output = None

for i in range(benchmark_runs):
    transaction = benchmark_input.copy()
    transaction["transaction_id"] = f"BENCHMARK_{i:06d}"

    start = time.perf_counter()

    output = call_detection(
        detection_function,
        transaction
    )

    end = time.perf_counter()

    if sample_output is None:
        sample_output = output

    latencies_ms.append(
        (end - start) * 1000
    )

latencies_ms = np.asarray(
    latencies_ms,
    dtype=float
)

results = {
    "component": "Full detection pipeline",
    "function": detection_function.__name__,
    "warmup_runs": warmup_runs,
    "benchmark_runs": benchmark_runs,
    "mean_ms": float(np.mean(latencies_ms)),
    "median_ms": float(np.median(latencies_ms)),
    "p95_ms": float(np.percentile(latencies_ms, 95)),
    "p99_ms": float(np.percentile(latencies_ms, 99)),
    "min_ms": float(np.min(latencies_ms)),
    "max_ms": float(np.max(latencies_ms))
}

print("FULL DETECTION PIPELINE")
print("=" * 60)
print(json.dumps(results, indent=2))
print("=" * 60)
print("Actual production output:")
print(json.dumps(sample_output, indent=2, default=str))

FULL DETECTION PIPELINE
{
  "component": "Full detection pipeline",
  "function": "get_ml_score",
  "warmup_runs": 20,
  "benchmark_runs": 1000,
  "mean_ms": 4.1202853999893705,
  "median_ms": 3.7263499998516636,
  "p95_ms": 5.634839999856922,
  "p99_ms": 7.2749789999124905,
  "min_ms": 2.992100000483333,
  "max_ms": 45.213999999759835
}
Actual production output:
{
  "ml_score": 0.0299536045640707,
  "ml_prediction": 0,
  "ml_label": "Legitimate",
  "model": "XGBoost",
  "model_version": "2.0"
}


In [17]:
import json

if "sample_output" not in globals():
    raise RuntimeError(
        "Run Cell 7 before running Cell 8."
    )

print("ACTUAL DETECTION SERVICE RETURN VALUE")
print("=" * 60)
print(json.dumps(sample_output, indent=2, default=str))
print("=" * 60)
print(f"Return type: {type(sample_output).__name__}")

ACTUAL DETECTION SERVICE RETURN VALUE
{
  "ml_score": 0.0299536045640707,
  "ml_prediction": 0,
  "ml_label": "Legitimate",
  "model": "XGBoost",
  "model_version": "2.0"
}
Return type: dict


In [18]:
from pathlib import Path
import sys
import inspect
import time

current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [current_path] + list(current_path.parents)
        if (path / "src").is_dir() and (path / "models").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Fraud-detection-ML-V2 project root was not found."
    )

SRC_DIR = PROJECT_ROOT / "src"

for path in [str(PROJECT_ROOT), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

import detection_service

public_functions = {
    name: obj
    for name, obj in inspect.getmembers(
        detection_service,
        inspect.isfunction
    )
    if not name.startswith("_")
}

detection_function = None

for name in [
    "detect_transaction",
    "detect",
    "process_transaction",
    "run_detection",
    "process_detection"
]:
    if name in public_functions:
        detection_function = public_functions[name]
        break

if detection_function is None:
    candidates = [
        obj
        for name, obj in public_functions.items()
        if any(
            word in name.lower()
            for word in ["detect", "process", "score"]
        )
    ]

    if candidates:
        detection_function = candidates[0]

if detection_function is None:
    raise RuntimeError(
        "Could not identify the production detection function."
    )

def call_detection(function_obj, values):
    signature = inspect.signature(function_obj)
    parameters = list(signature.parameters.values())

    if len(parameters) == 1:
        parameter = parameters[0]

        if parameter.name in values:
            return function_obj(values[parameter.name])

        return function_obj(values)

    kwargs = {}

    for parameter in parameters:
        if parameter.kind in (
            inspect.Parameter.VAR_POSITIONAL,
            inspect.Parameter.VAR_KEYWORD
        ):
            continue

        if parameter.name in values:
            kwargs[parameter.name] = values[parameter.name]
        elif parameter.default is inspect.Parameter.empty:
            raise ValueError(
                f"Required parameter '{parameter.name}' "
                f"could not be supplied."
            )

    return function_obj(**kwargs)

base_input = {
    "transaction_id": "BATCH_000000",
    "user_id": "USER_000001",
    "amount": 100.0,
    "amount_vs_avg_ratio": 1.0,
    "txn_count_last_5min": 0,
    "time_since_last_txn_sec": 600.0,
    "distance_from_last_location_km": 0.0,
    "merchant_category_is_new_for_user": 0
}

batch_size = 1000

batch_outputs = []

start = time.perf_counter()

for i in range(batch_size):
    transaction = base_input.copy()

    transaction["transaction_id"] = f"BATCH_{i:06d}"
    transaction["amount"] = float(
        100 + (i % 20) * 10
    )
    transaction["amount_vs_avg_ratio"] = float(
        1.0 + (i % 5) * 0.5
    )
    transaction["txn_count_last_5min"] = int(
        i % 6
    )
    transaction["time_since_last_txn_sec"] = float(
        60 + (i % 10) * 30
    )
    transaction["distance_from_last_location_km"] = float(
        i % 50
    )
    transaction["merchant_category_is_new_for_user"] = int(
        i % 2
    )

    output = call_detection(
        detection_function,
        transaction
    )

    batch_outputs.append(output)

end = time.perf_counter()

total_time_sec = end - start

throughput = (
    batch_size / total_time_sec
    if total_time_sec > 0
    else 0.0
)

average_time_ms = (
    total_time_sec / batch_size
) * 1000

print("BATCH THROUGHPUT")
print("=" * 60)
print(f"Batch size: {batch_size}")
print(f"Total time: {total_time_sec:.6f} seconds")
print(f"Throughput: {throughput:.2f} transactions/second")
print(f"Average transaction time: {average_time_ms:.4f} ms")

BATCH THROUGHPUT
Batch size: 1000
Total time: 3.960184 seconds
Throughput: 252.51 transactions/second
Average transaction time: 3.9602 ms


In [19]:
if "batch_outputs" not in globals():
    raise RuntimeError(
        "Run the batch throughput cell before this cell."
    )

expected_batch_size = 1000
actual_batch_size = len(batch_outputs)

print("BATCH EXECUTION CHECK")
print("=" * 60)
print(f"Expected outputs: {expected_batch_size}")
print(f"Actual outputs: {actual_batch_size}")

if actual_batch_size != expected_batch_size:
    raise RuntimeError(
        "Batch execution count does not match the requested batch size."
    )

print("Status: PASS")

BATCH EXECUTION CHECK
Expected outputs: 1000
Actual outputs: 1000
Status: PASS


In [20]:
from pathlib import Path
import json

current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [current_path] + list(current_path.parents)
        if (path / "src").is_dir() and (path / "models").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Fraud-detection-ML-V2 project root was not found."
    )

METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

phase12_results = {
    "phase": 12,
    "title": "Performance & Latency Benchmarking",
    "project_version": "V2.1",
    "full_detection_pipeline": globals().get("results"),
    "batch_throughput": {
        "batch_size": globals().get("batch_size"),
        "total_time_sec": globals().get("total_time_sec"),
        "throughput_transactions_per_second": globals().get("throughput"),
        "average_transaction_time_ms": globals().get("average_time_ms")
    }
}

output_path = (
    METRICS_DIR /
    "phase12_performance_benchmark.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        phase12_results,
        file,
        indent=2,
        default=str
    )

print(f"Saved benchmark results:")
print(output_path)

Saved benchmark results:
C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\outputs\metrics\phase12_performance_benchmark.json


In [21]:
from pathlib import Path

current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [current_path] + list(current_path.parents)
        if (path / "src").is_dir() and (path / "models").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Fraud-detection-ML-V2 project root was not found."
    )

benchmark_file = (
    PROJECT_ROOT /
    "outputs" /
    "metrics" /
    "phase12_performance_benchmark.json"
)

checks = {
    "project_root_found": PROJECT_ROOT.exists(),
    "full_pipeline_benchmark_completed": (
        isinstance(globals().get("results"), dict)
        and globals()["results"].get("benchmark_runs") == 1000
    ),
    "batch_completed": (
        len(globals().get("batch_outputs", [])) == 1000
    ),
    "benchmark_file_saved": benchmark_file.exists()
}

overall = all(checks.values())

print("PHASE 12 FINAL STATUS")
print("=" * 60)

for name, status in checks.items():
    print(f"{name}: {status}")

print("=" * 60)
print(f"Overall verification: {overall}")

if overall:
    print("PHASE 12 STATUS: READY")

PHASE 12 FINAL STATUS
project_root_found: True
full_pipeline_benchmark_completed: True
batch_completed: True
benchmark_file_saved: True
Overall verification: True
PHASE 12 STATUS: READY
